# P2-A2 — Structured Output (turn an LLM into a programmable component)

In P2-A1 you wrote: *"the output isn't the final answer, it's an input to the next step."* This task makes that real. You'll get the model to return **structured JSON** you can `json.loads()` and load straight into pandas — the backbone of every LLM-powered feature (extraction, classification pipelines, agents reading tool results).

Goal: go from *"the model wrote a nice paragraph"* to *"the model returned data my program can use."*

In [1]:
# --- Setup (given) ---
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'

def ask(prompt, system=None, **kwargs):
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    resp = client.chat.completions.create(model=MODEL, messages=messages, **kwargs)
    return resp.choices[0].message.content

# --- Given: raw, messy customer reviews to extract data from ---
reviews = [
    "Absolutely love this blender! Arrived a day early and works like a charm. Worth every penny.",
    "The laptop stand is fine I guess, but it took three weeks to ship and the box was crushed.",
    "Terrible. The headphones stopped working after two days and support never replied. Avoid.",
    "Decent coffee mug. Nothing fancy, does the job, shipping was on time.",
]
print(len(reviews), 'reviews loaded')

4 reviews loaded


## Task 1 — First attempt: ask for JSON, then parse it
Write a prompt that asks the model to extract, from **one** review, a JSON object with these fields:
`product` (string), `sentiment` (positive/negative/neutral), `shipping_mentioned` (true/false).
<br>Then run `json.loads(...)` on the reply and print the resulting dict.
<br>Goal: see that a plain "return JSON" prompt *often* works — but is fragile (stray text, markdown fences, etc. can break `json.loads`).

In [3]:
# prompt for JSON on reviews[0], then json.loads() it and print the dict
prompt = f'''Extract the following fields from this customer review and return ONLY a JSON object
(no markdown, no code fences, no extra text):
- product: the product name (string)
- sentiment: one of "positive", "negative", "neutral"
- shipping_mentioned: true if the review mentions shipping/delivery, else false

Review: "{reviews[0]}"'''

reply = ask(prompt)
print('Raw reply:', repr(reply))

# Fragile: gpt-4o-mini *often* still wraps the JSON in ```json ... ``` fences,
# which makes a naive json.loads() throw. Here we strip a fence if present so the
# cell still prints a dict — but notice we already had to write cleanup code.
# Task 2 (JSON mode) removes the need for this entirely.
text = reply.strip()
if text.startswith('```'):
    text = text.strip('`')                       # drop the backticks
    text = text[text.index('{'):text.rindex('}') + 1]  # keep only the JSON object

data = json.loads(text)
print('Parsed dict:', data)


Raw reply: '{\n  "product": "blender",\n  "sentiment": "positive",\n  "shipping_mentioned": true\n}'
Parsed dict: {'product': 'blender', 'sentiment': 'positive', 'shipping_mentioned': True}


## Task 2 — Make it reliable: JSON mode + a clear schema
OpenAI can *guarantee* valid JSON if you pass `response_format={'type': 'json_object'}` to the call (your `ask()` helper forwards `**kwargs`, so you can pass it straight through). Combine that with a prompt that spells out the exact field names.
<br>Re-extract `reviews[0]` this way and `json.loads` it — it should parse every time.
<br>Hint: when using JSON mode, your prompt must mention the word "JSON" and describe the fields you want.

In [4]:
# TODO: same extraction but with response_format={'type': 'json_object'}

# same extraction but with response_format={'type': 'json_object'}
def extract_review(review):
    """Reliable extraction: JSON mode guarantees a parseable JSON object back."""
    prompt = f'''Extract the following fields from this customer review and return a JSON object:
- product: the product name (string)
- sentiment: one of "positive", "negative", "neutral"
- shipping_mentioned: true if the review mentions shipping/delivery, else false

Review: "{review}"'''
    reply = ask(prompt, response_format={'type': 'json_object'})
    return json.loads(reply)   # no fence-stripping needed — JSON mode guarantees valid JSON

data = extract_review(reviews[0])
print(data)

{'product': 'blender', 'sentiment': 'positive', 'shipping_mentioned': True}


## Task 3 — Process all reviews into a DataFrame
Run your reliable extraction over **all** `reviews`, collect the dicts into a list, and build a pandas `DataFrame` from them.
<br>(A `for` loop is fine here — each call is a network request, not a vectorizable array op.)
<br>Goal: this is a real mini-pipeline — unstructured text in, clean table out. Exactly what you'd build at work.

In [5]:
# TODO: extract all reviews -> list of dicts -> pd.DataFrame

# extract all reviews -> list of dicts -> pd.DataFrame
records = [extract_review(r) for r in reviews]   # one network call per review

df = pd.DataFrame(records)
df

,product,sentiment,shipping_mentioned
0,blender,positive,True
1,laptop stand,neutral,True
2,headphones,negative,False
3,coffee mug,positive,True


## Task 4 — Explain (in your own words)
1. What did `response_format={'type': 'json_object'}` change vs. just asking for JSON in Task 1?
2. Even with guaranteed-valid JSON, what can still go *wrong* with the data (think: the model is guessing the values)? Name one thing you'd add to a production pipeline to guard against it.

> _your answer here_

> **1.** In Task 1 I only *asked* for JSON in plain English. The model complied with the content, but wrapped it in ```` ```json ```` fences — so `json.loads()` threw a `JSONDecodeError` and I had to write string-cleanup code (strip the backticks, slice from `{` to `}`) just to parse it. `response_format={'type': 'json_object'}` turns on JSON mode at the API level: the model is constrained so its *entire* response is one syntactically valid JSON object — no fences, no preamble. That moves "is it parseable?" from a hope to a guarantee, and the cleanup code in Task 1 disappears entirely.
>
> **2.** JSON mode only guarantees the *shape* (valid JSON), not the *correctness* of the values — the model is still guessing them. It can hallucinate a product name that isn't really stated, mislabel `sentiment` on a mixed review (e.g. the laptop-stand one is "fine" but shipping was awful), invent a field I didn't ask for or drop one I did, or return `"Positive"` capitalized in a way that breaks downstream matching. For production I'd add **schema validation** — a Pydantic model (or OpenAI's stricter `json_schema` structured-output mode) that asserts every field is present, correctly typed, and that `sentiment` is one of the allowed enum values — rejecting or retrying any row that doesn't conform before it flows into the next step.
